In [2]:
import torch
import torch.nn as nn

In [3]:
with open('/Users/manasa.mylavarapu/Desktop/AK_Practice/Transform_her/input.txt', 'r') as f:
    text = f.read()

In [4]:
chars = sorted(set(text))
vocab_size = len(chars)

In [5]:
stoi = {ch : i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

In [6]:
stoi
# itos

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

In [7]:
def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

In [8]:
print(vocab_size)
print(chars)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [9]:
data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [10]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data))
print(len(val_data))


1003853
111540


In [11]:
block_size = 8
x = train_data[:block_size]
y = train_data[1: block_size+1]
for t in range(block_size):
    context = x[: t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is {target}")


when input is [18] the target is 47
when input is [18, 47] the target is 56
when input is [18, 47, 56] the target is 57
when input is [18, 47, 56, 57] the target is 58
when input is [18, 47, 56, 57, 58] the target is 1
when input is [18, 47, 56, 57, 58, 1] the target is 15
when input is [18, 47, 56, 57, 58, 1, 15] the target is 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target is 58


In [12]:

batch_size = 4
def get_batch(split):
    data_ = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_)- block_size, (batch_size,))
    x_ = torch.stack([data_[i: i+block_size] for i in ix])
    y_ = torch.stack([data_[i+1: i+1+block_size] for i in ix])   
    return x_, y_
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('outputs:')
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target : {target}")

inputs:
torch.Size([4, 8])
tensor([[ 0, 13, 52, 42,  1, 45, 47, 60],
        [ 1, 52, 43, 60, 43, 56,  1, 45],
        [58, 41, 46,  8,  0,  0, 19, 39],
        [ 1, 48, 59, 57, 58, 47, 44, 63]])
outputs:
tensor([[13, 52, 42,  1, 45, 47, 60, 43],
        [52, 43, 60, 43, 56,  1, 45, 56],
        [41, 46,  8,  0,  0, 19, 39, 56],
        [48, 59, 57, 58, 47, 44, 63,  1]])
when input is [0] the target : 13
when input is [0, 13] the target : 52
when input is [0, 13, 52] the target : 42
when input is [0, 13, 52, 42] the target : 1
when input is [0, 13, 52, 42, 1] the target : 45
when input is [0, 13, 52, 42, 1, 45] the target : 47
when input is [0, 13, 52, 42, 1, 45, 47] the target : 60
when input is [0, 13, 52, 42, 1, 45, 47, 60] the target : 43
when input is [1] the target : 52
when input is [1, 52] the target : 43
when input is [1, 52, 43] the target : 60
when input is [1, 52, 43, 60] the target : 43
when input is [1, 52, 43, 60, 43] the target : 56
when input is [1, 52, 43, 60, 43, 56]

In [13]:
import torch
table = nn.Embedding(5, 5)
print(table.weight.shape)    # torch.Size([5, 5])
print(table.weight)

torch.Size([5, 5])
Parameter containing:
tensor([[ 0.2630,  0.5944,  1.5565,  0.2927, -1.4847],
        [ 0.6979,  0.9121,  1.2207, -1.1525, -0.7429],
        [-0.4511,  0.7832, -0.5846,  0.3233,  1.1273],
        [-0.3028, -0.1258,  0.3614,  0.2781, -0.4116],
        [-0.3503,  0.0330, -0.3331,  2.1066,  1.3413]], requires_grad=True)


In [14]:
from torch.nn import functional as F
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) #(B, T, C) 
        #m(xb) └──> nn.Module.__call__(self, xb, yb)└──> BigramLanguageModel.forward(self, idx=xb)
        if targets is None:
            loss = None  
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # logits (B, T,C)
            probs = F.softmax(logits[:, -1, :], dim = -1) #probs (B,C)
            # print("probs: and the shape")
            # print(probs)
            # print(probs.shape)
            idx_next = torch.multinomial(probs, num_samples=1) 
            # print("idx_next")
            # print(idx_next)
            idx = torch.cat((idx, idx_next), dim = 1)
            # print("idx: and shape")
            # print(idx)
            # print(idx.shape)
        return idx
        
        
        
m = BigramLanguageModel(vocab_size)
idx = torch.zeros((1,1), dtype=torch.long)
out = m.generate(idx, max_new_tokens=50)
print(decode(out[0].tolist()))
# logits, loss = m(xb, yb)
# print(logits)
# print(logits.shape)
# print(loss)




SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHj


In [15]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [16]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb,  yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() # computes the gradient (computes dJ/dw)
    optimizer.step() #calculates the new gradient (wt = wt-1 - alpha*dJ/dw)
print(loss.item())

2.4736921787261963


In [17]:
# idx = torch.zeros((1,1), dtype=torch.long)
# out = m.generate(idx, max_new_tokens=50)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Thando o an Thamyo ptithee thoud te.
BYotherquar ngifows ht IUS:
S:

ING flvenje ssutefr,
M:
War cl igagimous pray whars:
Panalit I It aithit terised the. be fonau buaror VOubed spo mng as chathab llll:
Ware,

ee
Thy,
Thooured aly y hindr's.
Fashat-owhrees s, share hathure Anfaneof f s llon!

ICLiroushanot

Then
Magend cugss, be jollrty

AROUFLom, ifay wil wher, gheatalloult llats howheprshakengayoref f f abighine ck orors n s?
NGABerd Foutheig vemy.
NG t isoststor hnor 'myougorme whe s'car n r 


In [18]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [19]:
xbow = torch.zeros(B,T,C)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [20]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim= True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [26]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei @ x
xbow3[0]


tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])